In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import splink

%matplotlib inline

os.chdir(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa')

df_l = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\raw\df_l_censo_sin_snsa.parquet')
df_r = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\data\raw\df_r_rraa_sin_snsa.parquet')

## Reglas de bloqueo

In [2]:
from splink.duckdb.linker import DuckDBLinker
from splink.duckdb.blocking_rule_library import block_on

settings = {"link_type": "link_only"}

linker_br = DuckDBLinker([df_r, df_l], settings)

br1_a = block_on(["documento"])
#br1_b = block_on(["documento_extranjero"])
#br2 = block_on(["documento", "documento_extranjero", "fecha_nacimiento"])
br3 = block_on(["fecha_nacimiento", "primer_nombre", "primer_apellido"])
#br4 = block_on(["primer_nombre", "primer_apellido"]) (433,314,151)
br5 = block_on(["primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido"])
#br6 = block_on(["primer_nombre", "primer_apellido", "id_sexo"]) (430,033,729)
#br7 = block_on(["fecha_nacimiento", "id_sexo"]) (231,723,037)
br8 = block_on(["ano_nacimiento", "primer_nombre", "primer_apellido"])
br9 = block_on(["mes_nacimiento", "dia_nacimiento", "primer_nombre", "primer_apellido"])
#br10 = block_on(["id_departamento", "primer_nombre", "primer_apellido"]) (63,427,012)
#br11 = block_on(["id_departamento", "ano_nacimiento"]) (29,316,195,010)
br12 = block_on(["id_departamento", "fecha_nacimiento"]) #(83,471,400)
#br13 = block_on(["id_departamento", "substr(primer_nombre, 1,1)", "substr(primer_apellido, 1,1)"]) (13,581,242,609)
#br14 = block_on(["id_departamento", "primer_nombre", "ano_nacimiento"]) (292,595,639)
#br15 = block_on(["id_departamento", "substr(primer_nombre, 1,1)", "substr(primer_apellido, 1,1)", "ano_nacimiento"]) (161,287,979)
br16 = block_on(["id_departamento", "primer_nombre", "ano_nacimiento", "substr(primer_apellido, 1, 1)"])
br17 = block_on(["id_departamento_ajustado", "primer_nombre", "ano_nacimiento", "substr(primer_apellido, 1, 1)"])
br18 = block_on(["id_departamento_ajustado", "fecha_nacimiento"])

list_br = [br1_a, br3, br5, br8, br9, br12, br16, br17, br18]

for br in list_br:
    count = linker_br.count_num_comparisons_from_blocking_rule(br)
    print(f"Comparisons generated by '{br.blocking_rule_sql}': {count:,.0f}")

### Modelo 1

In [22]:
blocking_rules = [br1_a, br5, br8, br17, br18]

linker_br.cumulative_num_comparisons_from_blocking_rules_chart(blocking_rules)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.Chart(...)

## Definición de las comparaciones

El siguiente paso en la construcción del modelo es definir las métricas para comparar cada variable. 

In [3]:
from scripts.comparaciones import get_comparacion_fecha_nacimiento, get_comparacion_ano_nacimiento_imputados, get_comparacion_nombres_combinados, get_comparacion_nombre
import splink.duckdb.comparison_library as cl
import splink.duckdb.comparison_template_library as ctl
import splink.duckdb.comparison_level_library as cll

comparacion_fecha_nacimiento = get_comparacion_fecha_nacimiento(con_dc = True)
comparacion_ano_nacimiento_imputados = get_comparacion_ano_nacimiento_imputados()

comparacion_sexo = cl.exact_match("id_sexo")
comparacion_documento = cl.damerau_levenshtein_at_thresholds("documento", [1,2])
#comparacion_documento_extranjero = cl.levenshtein_at_thresholds("documento_extranjero", [1])
#comparacion_id_pais_documento = cl.exact_match("id_pais_documento")
#comparacion_id_tipo_documento = cl.exact_match("id_tipo_documento")

comparacion_departamento = cl.exact_match("id_departamento", term_frequency_adjustments = True)
comparacion_departamento_ajustado = cl.exact_match("id_departamento_ajustado", term_frequency_adjustments = True)

comparacion_nombres = get_comparacion_nombres_combinados("primer_nombre", "segundo_nombre", "nombres")
comparacion_apellidos = get_comparacion_nombres_combinados("primer_apellido", "segundo_apellido", "apellidos")

comparacion_primer_nombre = get_comparacion_nombre("primer_nombre", usar_tfa = True)
comparacion_segundo_nombre = get_comparacion_nombre("segundo_nombre", usar_tfa = True)
comparacion_primer_apellido = get_comparacion_nombre("primer_apellido", usar_tfa = True)
comparacion_segundo_apellido = get_comparacion_nombre("segundo_apellido", usar_tfa = True)

In [4]:
settings = {
    "link_type": "link_only",
    "comparisons": [
        comparacion_documento,
        comparacion_fecha_nacimiento,
        comparacion_ano_nacimiento_imputados,
        comparacion_primer_nombre,
        comparacion_segundo_nombre,
        comparacion_primer_apellido,
        comparacion_segundo_apellido,
        #comparacion_sexo,
        comparacion_departamento_ajustado
    ],
    "blocking_rules_to_generate_predictions": [
        br1_a,
        br5,
        br8,
        br17,
        br18
    ],
    "em_convergence": 1e-8,
    "max_iterations": 25,
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = DuckDBLinker([df_l, df_r], settings)

# Estimación de los parámetros

## 1. Probabilidad de que dos individuos sean match

In [5]:
p1 = 1/len(df_l)
p2 = 1/3500000

print("Probabilidad 'teórica' de que dos individuos sean match (población censada):", p1)
print("Probabilidad 'teórica' de que dos individuos sean match (población estimada):", p2)

deterministic_rules = [
    "l.documento = r.documento"
]

linker.estimate_probability_two_random_records_match(deterministic_rules, recall = 0.9)

Probabilidad 'teórica' de que dos individuos sean match (población censada): 3.1903432139326117e-07
Probabilidad 'teórica' de que dos individuos sean match (población estimada): 2.857142857142857e-07


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Probability two random records match is estimated to be  1.65e-07.
This means that amongst all possible pairwise record comparisons, one in 6,049,405.82 are expected to match.  With 19,209,578,998,385 total possible comparisons, we expect a total of around 3,175,448.89 matching pairs


## 2. Probabilidad de vincular dos observaciones dado que NO son match (u)

In [6]:
linker.estimate_u_using_random_sampling(max_pairs=1e8)

----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - documento (no m values are trained).
    - fecha_nacimiento (no m values are trained).
    - ano_nacimiento_imputados (no m values are trained).
    - primer_nombre (no m values are trained).
    - segundo_nombre (no m values are trained).
    - primer_apellido (no m values are trained).
    - segundo_apellido (no m values are trained).
    - id_departamento_ajustado (no m values are trained).


## 3. Probabilidad de vincular dos observaciones dado que son match (m)

In [7]:
linker.debug_mode = False

t_br1 = block_on(["documento"])
training_session1 = linker.estimate_parameters_using_expectation_maximisation(t_br1)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."documento" = r."documento"

Parameter estimates will be made for the following comparison(s):
    - fecha_nacimiento
    - ano_nacimiento_imputados
    - primer_nombre
    - segundo_nombre
    - primer_apellido
    - segundo_apellido
    - id_departamento_ajustado

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - documento



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Iteration 1: Largest change in params was 0.524 in probability_two_random_records_match


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Iteration 2: Largest change in params was 0.00653 in the m_probability of ano_nacimiento_imputados, level `All other comparisons`


...

Iteration 6: Largest change in params was 2.97e-07 in probability_two_random_records_match


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Iteration 7: Largest change in params was 4.11e-08 in probability_two_random_records_match


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Iteration 8: Largest change in params was 5.69e-09 in probability_two_random_records_match

EM converged after 8 iterations

Your model is not yet fully trained. Missing estimates for:
    - documento (no m values are trained).


In [23]:
t_br2 = block_on(["primer_nombre", "segundo_nombre", "primer_apellido", "segundo_apellido", "fecha_nacimiento"])
training_session_dob = linker.estimate_parameters_using_expectation_maximisation(t_br2)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."primer_nombre" = r."primer_nombre") AND (l."segundo_nombre" = r."segundo_nombre") AND (l."primer_apellido" = r."primer_apellido") AND (l."segundo_apellido" = r."segundo_apellido") AND (l."fecha_nacimiento" = r."fecha_nacimiento")

Parameter estimates will be made for the following comparison(s):
    - documento
    - ano_nacimiento_imputados
    - id_departamento_ajustado

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - fecha_nacimiento
    - primer_nombre
    - segundo_nombre
    - primer_apellido
    - segundo_apellido

Level Levenshtein <= 1 on comparison ano_nacimiento_imputados not observed in dataset, unable to train m value

Level All other comparisons on comparison ano_nacimiento_imputados not observed in dataset, unable to train m value

Iteration 1: Largest change in params was 0.298 in the m_probabilit

## 4. Análisis de los match weights

In [8]:
linker.match_weights_chart()

alt.VConcatChart(...)

In [27]:
linker.tf_adjustment_chart("id_departamento_ajustado")

c:\Users\lpescetto\AppData\Local\Programs\Python\Python312\Lib\site-packages\splink\linker.py:3591: UserWarning: Values [None] from `vals_to_include` were not found in the dataset so are not included in the chart.
  return tf_adjustment_chart(
c:\Users\lpescetto\AppData\Local\Programs\Python\Python312\Lib\site-packages\splink\term_frequencies.py:259: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["gamma", "bin", "log2_bf"])


alt.HConcatChart(...)

In [28]:
linker.tf_adjustment_chart("primer_nombre")

c:\Users\lpescetto\AppData\Local\Programs\Python\Python312\Lib\site-packages\splink\linker.py:3591: UserWarning: Values [None] from `vals_to_include` were not found in the dataset so are not included in the chart.
  return tf_adjustment_chart(
c:\Users\lpescetto\AppData\Local\Programs\Python\Python312\Lib\site-packages\splink\term_frequencies.py:259: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["gamma", "bin", "log2_bf"])


alt.HConcatChart(...)

In [14]:
linker.tf_adjustment_chart("segundo_nombre")

c:\Users\lpescetto\AppData\Local\Programs\Python\Python312\Lib\site-packages\splink\linker.py:3591: UserWarning: Values [None] from `vals_to_include` were not found in the dataset so are not included in the chart.
  return tf_adjustment_chart(
c:\Users\lpescetto\AppData\Local\Programs\Python\Python312\Lib\site-packages\splink\term_frequencies.py:259: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["gamma", "bin", "log2_bf"])


alt.HConcatChart(...)

In [12]:
linker.tf_adjustment_chart("segundo_apellido")

c:\Users\lpescetto\AppData\Local\Programs\Python\Python312\Lib\site-packages\splink\linker.py:3591: UserWarning: Values [None] from `vals_to_include` were not found in the dataset so are not included in the chart.
  return tf_adjustment_chart(
c:\Users\lpescetto\AppData\Local\Programs\Python\Python312\Lib\site-packages\splink\term_frequencies.py:259: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["gamma", "bin", "log2_bf"])


alt.HConcatChart(...)

In [29]:
linker.m_u_parameters_chart()

alt.HConcatChart(...)

In [30]:
linker.unlinkables_chart()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.LayerChart(...)

In [31]:
settings = linker.save_model_to_json(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\modelos\modelo4.json', overwrite=False)